# 01 — Download the datasets

This notebook stages Stage IV daily precipitation, the 2006-2020 precipitation normals, gridMET wind, GRACE-DA, HLS for August and the CSB field polys for Iowa, for the AOI and `SEASONS`.
A re-run skips valid files and re-fetches only missing or broken ones; HLS and GRACE-DA need the Earthdata secret.

In [0]:
%run ./config_nb

geobrix 0.6.0: one wheel per runtime (+env5 / +env6); SQL functions register through databricks.labs.gbx.register(spark); the Python API is rasterx / gridx / vectorx. See https://databrickslabs.github.io/geobrix/docs/migration-0-6
geobrix: JVM level unavailable: Spark Connect session (Serverless); using the Python level (48 functions)
geobrix: SQL expression level unavailable: 5 functions (gbx_bng_cellarea, gbx_bng_eastnorthasbng, gbx_quadbin_distance, ...) need a SQL function wrapper, and SQL functions are unavailable here; using the Python level


... EARTHDATA_TOKEN loaded from geospatial_docs.vapor_eyes.earthdata_token


True

DataFrame[]

H3PX_DIR = /Volumes/geospatial_docs/h3_pixels_crop_yield/data/h3-pixels-crop-yield


... view helpers: show_cells, show_tile (gated by INTERACTIVE_PLOTS)


## Stage every dataset

The bbox does not depend on the season or the field selection, so one download serves every season.
CSB is one national file, clipped to Iowa once.

In [0]:
bbox = get_aoi_bbox()
frames = []


def _tag(df, dataset):
    return df.withColumn("dataset", F.lit(dataset))


for season in SEASONS:
    print(f"... season {season}: Stage IV, gridMET")
    frames.append(_tag(stage4.download([season], MONTHS["stage4"], STAGE4_DIR, force=FORCE_REBUILD, spark=spark), "stage4"))
    frames.append(_tag(gridmet.download([season], GRIDMET_DIR, force=FORCE_REBUILD, spark=spark), "gridmet"))
    if EARTHDATA_READY:
        print(f"... season {season}: GRACE-DA, HLS (August)")
        frames.append(_tag(grace.download(bbox, GRACE_DIR, temporal=(f"{season}-05-01", f"{season}-09-30"),
                                          force=FORCE_REBUILD, spark=spark), "grace"))
        frames.append(_tag(hls.download(bbox, HLS_DIR, temporal=(f"{season}-08-01", f"{season}-08-31"),
                                        force=FORCE_REBUILD, spark=spark), "hls"))
print("... precipitation normals 2006-2020")
frames.append(_tag(nclim.normals(NORMALS_DIR, force=FORCE_REBUILD, spark=spark), "normals"))
if USE_NCLIMGRID:
    frames.append(_tag(nclim.monthly(NCLIMGRID_DIR, force=FORCE_REBUILD, spark=spark), "nclimgrid"))
print("... CSB field polys, clipped to Iowa")
frames.append(_tag(csb.download(CSB_DIR, state_fips="19", force=FORCE_REBUILD, spark=spark), "csb"))

cols = ["dataset", "item_id", "asset_name", "out_file_path", "out_file_sz", "is_out_file_valid", "last_update"]
downloads = _persist(functools.reduce(DataFrame.unionByName, frames).select(cols), "downloads", rewrite=True)

... season 2023: Stage IV, gridMET
... season 2023: GRACE-DA, HLS (August)
... precipitation normals 2006-2020
... CSB field polys, clipped to Iowa
... wrote table downloads (277 rows)


## What was staged

One row per dataset: files, valid files and size.

In [0]:
summary = spark.sql("""
  SELECT dataset, count(*) AS files, sum(CAST(is_out_file_valid AS INT)) AS valid,
         round(sum(out_file_sz) / 1e9, 2) AS gb
  FROM downloads GROUP BY dataset ORDER BY dataset""")
display(summary)
bad = spark.table("downloads").where("NOT is_out_file_valid")
if bad.count():
    display(bad)
assert bad.count() == 0, "some assets are invalid: rerun this notebook (valid files are kept) or call <downloader>.repair('downloads', ...)"
expected = {"stage4", "gridmet", "normals", "csb"} | ({"grace", "hls"} if EARTHDATA_READY else set())
expected |= {"nclimgrid"} if USE_NCLIMGRID else set()
assert expected <= {r["dataset"] for r in summary.collect()}

dataset,files,valid,gb
csb,1,1,0.51
grace,22,22,0.02
gridmet,1,1,0.06
hls,99,99,1.26
normals,1,1,0.28
stage4,153,153,0.62


## Check the HLS bands and the GRACE-DA week date

Each HLS scene gives red `B04`, NIR (`B05` for Landsat, `B8A` for Sentinel-2) and `Fmask`.
Each GRACE-DA file's first time value is the date in its name.

In [0]:
if EARTHDATA_READY:
    import netCDF4

    from databricks.labs.gbx.sample.grace import week_date_of

    # HLS: the staged assets of every scene are exactly red, NIR and Fmask, with the sensor's own NIR band.
    want = {"L30": {"red": "B04", "nir": "B05", "fmask": "Fmask"}, "S30": {"red": "B04", "nir": "B8A", "fmask": "Fmask"}}
    by_scene = {}
    for r in spark.table("downloads").where("dataset = 'hls'").select("item_id", "asset_name").collect():
        scene, band = r["item_id"].rsplit(".", 1)
        by_scene.setdefault(scene, {})[r["asset_name"]] = band
    bad = sorted(s for s, a in by_scene.items() if a != want[s.split(".")[1]])
    print(f"... HLS: {len(by_scene)} scenes, sensors {sorted({s.split('.')[1] for s in by_scene})}")
    assert by_scene and not bad, f"HLS scenes without the expected red/NIR/Fmask bands: {bad[:5]}"
    # GRACE-DA: the A<YYYYMMDD> stamp in each file name is the file's first time value.
    paths = [r["out_file_path"] for r in
             spark.table("downloads").where("dataset = 'grace' AND is_out_file_valid").select("out_file_path").collect()]
    for p in paths:
        with netCDF4.Dataset(p) as nc:
            t = nc.variables["time"]
            first = netCDF4.num2date(t[0], t.units, getattr(t, "calendar", "standard"))
        stamp = week_date_of(p)
        assert (first.year, first.month, first.day) == (stamp.year, stamp.month, stamp.day), (p, str(first))
    print(f"... GRACE-DA: {len(paths)} weekly files, each dated by its first time value")

... HLS: 33 scenes, sensors ['L30', 'S30']
... GRACE-DA: 22 weekly files, each dated by its first time value
